# Gaze gate, end to end: build the dataset, label it, train and test — 5-fold CV

One notebook, nothing prepared in advance. It downloads Aria Everyday Activities (AEA) videos,
builds the teacher signal with a frozen DINOv2, labels and splits the data, and trains a small
gate that decides **from the wearer's gaze alone** whether a frame is worth sending to a
vision-language model.

```
                  ┌──────────── teacher (training only) ────────────┐
frame t-1 ──► [DINOv2, frozen] ──► z_I0 ─┐
                                          ├─► h ──► ẑ_M ─┐
frame t   ──► [DINOv2, frozen] ──► z_IT ─┘               │ L_NCE  (contrastive)
                                                         │
gaze rates (3,9) ──► f_M ──► z_M ────────────────────────┘
                              │
                              └──► head ──► [S_frame, S_gaze] ──► L_sim  (regression to the teacher's scores)
```

The teacher sees pixels; the student (`f_M` + head) sees only gaze velocities. Only the student is
kept: at inference **no frame is encoded at all**.

## Run order

| section | what | time |
|---|---|---|
| 1 | clone the code, install | ~3 min |
| 2 | settings: how many videos, where to keep the data | — |
| 3 | your AEA download-links JSON | — |
| 4 | **build the dataset**: download → 1 fps frames → DINOv2 → one CSV row per frame pair | **~3–5 min per video** |
| 5 | verify the dataset | <1 min |
| 6–8 | label every row (4 classes), hold out a test set, draw 5 folds | <1 min |
| 9–10 | **train** `L_NCE + L_sim` over 5 folds | a few min |
| 11–12 | ablation (`L_sim` only) and a shuffle control that must fail | a few min |
| 13–14 | test once, through the SEND / SKIP rule | <1 min |
| 15 | export the trained gate as a standalone file | <1 min |

## Before you start

- **Runtime:** *Runtime → Change runtime type → T4 GPU* (DINOv2 runs on it during section 4).
- **AEA access:** request the dataset at [projectaria.com/datasets/aea](https://www.projectaria.com/datasets/aea/),
  accept its license, and download the **download-links JSON** it gives you. The links in it
  expire after about two weeks; download a fresh JSON if section 4 starts failing.
- **Disk:** each video is ~2.3 GB to download (video + eye gaze + VRS) and is deleted as soon as
  its features are written, so peak disk use stays at about one video.
- **Rerunning is safe.** Section 4 resumes where it stopped; with `SAVE_TO_DRIVE = True` it also
  survives a runtime reset.

## 1 — Clone the code and install

In [ ]:
!git clone -q -b main https://github.com/shubhamOjha1000/GazeVLM-HWSW-Codesign.git /content/GazeVLM
%cd /content/GazeVLM
!git log --oneline -1

!pip -q install -r requirements.txt
!pip -q install projectaria-tools

import os, sys, glob, json, time, shutil, random, subprocess
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt

REPO = "/content/GazeVLM"
sys.path.insert(0, REPO)

print("\ntorch", torch.__version__)
if torch.cuda.is_available():
    print("GPU  ", torch.cuda.get_device_name(0))
else:
    print("no GPU -- Runtime > Change runtime type > T4. Section 4 will be much slower on CPU.")

## 2 — Settings

| setting | meaning |
|---|---|
| `N_VIDEOS` | how many AEA videos to build. **At least 7** (5 folds + a test set); 12 is a good first run, 30+ gives steadier numbers |
| `SEED` | which videos are drawn. Same seed → same videos |
| `SAVE_TO_DRIVE` | `True` keeps the dataset and the results in your Google Drive, so a runtime reset does not mean downloading everything again |

Everything is built under `/content/lab` (fast local disk). With `SAVE_TO_DRIVE`, each finished
video is also copied to Drive and restored from there in the next session.

In [ ]:
N_VIDEOS      = 12
SEED          = 0
SAVE_TO_DRIVE = False
DRIVE_DIR     = "/content/drive/MyDrive/gaze_gate_lab"     # used only if SAVE_TO_DRIVE
MAX_FAILURES  = 5      # videos that fail to download are skipped; stop after this many

WORK       = "/content/lab"
RAW_DIR    = os.path.join(WORK, "raw")              # deleted video by video
FRAMES_DIR = os.path.join(WORK, "frames_1fps")      # deleted video by video
FEAT_DIR   = os.path.join(WORK, "features")         # <sequence>/feat_*.npz, kept
CSV        = os.path.join(WORK, "feature_dataset.csv")
CSV_LAB    = os.path.join(WORK, "feature_dataset_labeled.csv")
FOLD_DIR   = os.path.join(WORK, "folds")
THRESH     = os.path.join(WORK, "thresholds.json")
FEAT_CACHE = os.path.join(WORK, "feat_cache.npz")
RUNS_DIR   = os.path.join(WORK, "runs")
for d in (WORK, RAW_DIR, FRAMES_DIR, FEAT_DIR, FOLD_DIR, RUNS_DIR):
    os.makedirs(d, exist_ok=True)

assert N_VIDEOS >= 7, "need at least 7 videos: 5 folds of training videos plus held-out test videos"

PERSIST = None
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    PERSIST = DRIVE_DIR
    os.makedirs(os.path.join(PERSIST, "features"), exist_ok=True)
    saved = os.path.join(PERSIST, "feature_dataset.csv")
    if os.path.exists(saved) and not os.path.exists(CSV):
        prev = pd.read_csv(saved)
        kept_seqs = []
        for s in prev["sequence"].unique():
            tar = os.path.join(PERSIST, "features", f"{s}.tar")
            if os.path.exists(tar):
                shutil.unpack_archive(tar, FEAT_DIR)
                kept_seqs.append(s)
        prev[prev["sequence"].isin(kept_seqs)].to_csv(CSV, index=False)
        print(f"restored {len(kept_seqs)} videos from {PERSIST}")

print(f"working dir : {WORK}")
print(f"Drive copy  : {PERSIST or 'off'}")
free = shutil.disk_usage("/content").free / 1e9
print(f"free disk   : {free:.0f} GB   {'ok' if free > 15 else '<- under 15 GB; free some space'}")

## 3 — Your AEA download-links JSON

Upload the JSON you got from the AEA website (the file picker opens below). With
`SAVE_TO_DRIVE`, a copy is kept in Drive and used next time without uploading.

In [ ]:
kept = os.path.join(PERSIST, "aea_download_urls.json") if PERSIST else None
if kept and os.path.exists(kept):
    URLS_JSON = kept
    print("using the copy in Drive:", kept)
else:
    from google.colab import files
    up = files.upload()                                   # pick your download-links JSON
    name = list(up.keys())[0]
    URLS_JSON = os.path.join(WORK, "aea_download_urls.json")
    shutil.move(name, URLS_JSON)
    if kept:
        shutil.copy2(URLS_JSON, kept)

meta = json.load(open(URLS_JSON))
assert "sequences" in meta, "this does not look like the AEA download-links JSON (no 'sequences' key)"
need = ("video_main_rgb", "mps_eye_gaze", "main_vrs")
ok = [s for s, e in meta["sequences"].items() if all(k in e for k in need)]
print(f"{len(meta['sequences'])} videos in the JSON, {len(ok)} with video + eye gaze + VRS")
assert len(ok) >= N_VIDEOS, f"only {len(ok)} usable videos; lower N_VIDEOS"

# the candidate order: a seeded shuffle, so the same SEED always draws the same videos
CANDIDATES = sorted(ok)
random.Random(SEED).shuffle(CANDIDATES)
gb = sum(meta["sequences"][s][k]["file_size_bytes"] for s in CANDIDATES[:N_VIDEOS] for k in need) / 1e9
print(f"first {N_VIDEOS} candidates: {gb:.0f} GB to download in total (one video on disk at a time)")

## 4 — Build the dataset

For each video, the repo's builder (`src/dataprep/build_feature_csv.py`):

1. downloads the RGB video, the eye-gaze stream and the VRS (camera calibration + IMU);
2. takes **1 frame per second**;
3. runs a **frozen DINOv2** on every frame and saves its CLS token and patch grid to a small `.npz`;
4. writes **one row per pair of consecutive frames**: the two teacher scores and the gaze in between.

| column | per row | what |
|---|---|---|
| `sequence` | 1 | video name |
| `feat_frame_1`, `feat_frame_2` | 1 | the two frames' DINOv2 feature files |
| `frame_similarity` | 1 | cosine of the two CLS tokens — **teacher score 1**: did the whole picture change? |
| `gaze_patch_token_sim` | 1 | cosine of the patch tokens at the gaze point — **teacher score 2**: did what the wearer looks at change? |
| `gaze_rates_window` | 9 × 3 | eye yaw rate, pitch rate and their magnitude (rad/s) over the second — **the student's input** |
| `gaze_vec3d_rates_window` | 9 × 3 | d/dt of the 3D unit gaze vector (an exact-speed alternative) |
| `imu_window` | 10 × 6 | head accelerometer + gyroscope, binned to 10 Hz |

The raw video is deleted as soon as its features exist. **This is the long cell** (a few minutes
per video, mostly download). It checkpoints after every video, so rerunning it continues where it
stopped; a video whose download fails is skipped and the next candidate is used.

In [ ]:
def built():
    """Videos already in the dataset CSV."""
    return set(pd.read_csv(CSV, usecols=["sequence"])["sequence"]) if os.path.exists(CSV) else set()


def backup(seq):
    """Copy the CSV and this video's features to Drive (one tar per video)."""
    if PERSIST:                         # features first, so the saved CSV never names a missing video
        shutil.make_archive(os.path.join(PERSIST, "features", seq), "tar", FEAT_DIR, seq)
        shutil.copy2(CSV, os.path.join(PERSIST, "feature_dataset.csv"))


failed, t_start = [], time.time()
for seq in CANDIDATES:
    done = built()
    if len(done) >= N_VIDEOS:
        break
    if seq in done:
        continue
    if len(failed) >= MAX_FAILURES:
        print(f"\n!! {len(failed)} videos failed -- stopping. If they all failed to download, your "
              f"links have probably expired: get a fresh JSON and rerun sections 3 and 4.")
        break
    print(f"\n{'#' * 78}\n# video {len(done) + 1}/{N_VIDEOS}: {seq}   "
          f"({(time.time() - t_start) / 60:.0f} min so far)\n{'#' * 78}", flush=True)
    r = subprocess.run([sys.executable, "-m", "src.dataprep.build_feature_csv",
                        "--urls_json", URLS_JSON, "--out_csv", CSV,
                        "--raw_dir", RAW_DIR, "--frames_dir", FRAMES_DIR, "--feat_dir", FEAT_DIR,
                        "--seqs", seq, "--resume", "--imu_hz", "10", "--cleanup_raw"], cwd=REPO)
    if r.returncode != 0 or seq not in built():
        failed.append(seq)
        # a half-finished download would be reused as if complete, so throw this video away entirely
        for d in (RAW_DIR, FRAMES_DIR, FEAT_DIR):
            shutil.rmtree(os.path.join(d, seq), ignore_errors=True)
        print(f"!! {seq} failed (exit code {r.returncode}); trying the next candidate")
        continue
    backup(seq)

done = built()
print(f"\n{'=' * 78}\n{len(done)} videos built in {(time.time() - t_start) / 60:.0f} min"
      f"{'   failed: ' + ', '.join(failed) if failed else ''}")
assert len(done) >= 7, "fewer than 7 videos built -- rerun this cell (it resumes)"
if len(done) < N_VIDEOS:
    print(f"   (asked for {N_VIDEOS}; rerun this cell to add the rest, or continue with {len(done)})")

## 5 — Verify the dataset

Reads the CSV back, checks every feature file it points at exists, and **recomputes both teacher
scores from the saved features** for one row. If this passes, the dataset is self-contained.

In [ ]:
from src.loss1.dataset import parse_rates

data = pd.read_csv(CSV)
print(f"{len(data):,} rows x {data.shape[1]} columns from {data['sequence'].nunique()} videos")
display(data.groupby("sequence").size().rename("rows").to_frame().T)

paths = pd.concat([data.feat_frame_1, data.feat_frame_2]).unique()
missing = [p for p in paths if not os.path.exists(p)]

z0, z1 = np.load(data.loc[0, "feat_frame_1"]), np.load(data.loc[0, "feat_frame_2"])
fs = float(np.dot(z0["cls"], z1["cls"]))                 # tokens are L2-normalised
g = int(z0["grid"])
def cell(z):
    x, y = z["gaze_xy"]
    gx = min(g - 1, int(np.clip(x, 0, 1) * g)); gy = min(g - 1, int(np.clip(y, 0, 1) * g))
    return z["patches"][gy * g + gx]
ps = float(np.dot(cell(z0), cell(z1)))

n_rates = np.array([len(parse_rates(s)) for s in data["gaze_rates_window"]])
checks = [
    ("every feature file exists", len(missing) == 0, f"{len(missing)} missing of {len(paths):,}"),
    ("frame_similarity recomputes", abs(fs - data.loc[0, "frame_similarity"]) < 1e-3,
     f"{fs:.4f} vs {data.loc[0, 'frame_similarity']:.4f}"),
    ("gaze_patch_token_sim recomputes", abs(ps - data.loc[0, "gaze_patch_token_sim"]) < 1e-3,
     f"{ps:.4f} vs {data.loc[0, 'gaze_patch_token_sim']:.4f}"),
    ("gaze window has 9 steps", bool(np.median(n_rates) == 9), f"median {int(np.median(n_rates))} steps"),
    ("no NaN in the teacher scores", bool(data[["frame_similarity", "gaze_patch_token_sim"]].notna().all().all()), ""),
]
for name, ok_, detail in checks:
    print(f"   [{'PASS' if ok_ else 'FAIL'}]  {name:34s} {detail}")
assert all(c[1] for c in checks), "dataset check failed -- see above"

fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
ax[0].hist(data.frame_similarity, bins=60); ax[0].set_title("frame_similarity (teacher 1)")
ax[1].hist(data.gaze_patch_token_sim, bins=60, color="tab:orange"); ax[1].set_title("gaze_patch_token_sim (teacher 2)")
plt.tight_layout(); plt.show()

## 6 — Labels and splits

Each row gets one of four classes from the two teacher scores, each cut at a threshold τ:

| | gaze HIGH (same thing looked at) | gaze LOW (looked at something new) |
|---|---|---|
| **frame HIGH** (scene unchanged) | `STABLE` (3) — **SKIP** | `REFIXATION` (2) — SEND |
| **frame LOW** (scene changed) | `PURSUIT` (1) — SEND | `TRANSITION` (0) — SEND |

`quad = 2·frame_high + gaze_high`. Only `STABLE` frames are skipped.

**Two rules the split obeys.** Everything is grouped **by video** (consecutive rows are one second
apart and nearly identical, so a row-level split would put the same footage on both sides). And
**the test set is removed first** and never appears in any fold: fold validation is for choosing
settings, test is the one number you report, touched once.

Videos are sorted by mean `frame_similarity` (how busy the scene is); every 6th goes to test and
the rest are dealt round-robin into 5 folds, so every fold spans quiet and busy scenes.

In [ ]:
K          = 5      # number of CV folds
TEST_EVERY = 6      # every 6th video (in activity order) is held out for test

lite = pd.read_csv(CSV, usecols=["sequence", "frame_similarity", "gaze_patch_token_sim"])
order = lite.groupby("sequence")["frame_similarity"].mean().sort_values().index.tolist()

test_videos = order[::TEST_EVERY]
pool        = [v for v in order if v not in set(test_videos)]
assert len(pool) >= K, f"{len(pool)} training videos for {K} folds -- build more videos"
fold_of     = {v: i % K for i, v in enumerate(pool)}

lite["is_test"] = lite["sequence"].isin(test_videos).astype(int)
lite["fold"]    = lite["sequence"].map(fold_of).fillna(-1).astype(int)

print(f"{len(order)} videos -> {len(test_videos)} test + {len(pool)} pooled into {K} folds\n")
print(f"   TEST    {len(test_videos):3d} videos   {int(lite.is_test.sum()):6,} rows   "
      f"({100*lite.is_test.mean():4.1f}%)")
for k in range(K):
    m = lite.fold == k
    print(f"   fold {k}  {sum(v == k for v in fold_of.values()):3d} videos   "
          f"{int(m.sum()):6,} rows   ({100*m.mean():4.1f}%)")

train_pool = lite[lite.is_test == 0]          # thresholds come from here only; test is untouched
print(f"\nthresholds will use the {len(train_pool):,} pooled rows")

## 7 — Diagnostics: is the median the right cut?

1. **Is either score bimodal?** If yes, a data-driven cut (GMM) means something. If unimodal — the
   usual case for cosine similarity — the median is the honest choice.
2. **How much of a score is just *which video this is*?** The intraclass correlation (ICC) is
   between-video variance ÷ total. Above ~0.4 a global threshold mostly encodes scene identity.
3. **Does the joint distribution leave `REFIXATION` usable?** Median cuts give 50/50 on each axis
   but say nothing about the four cells. The more correlated the two scores, the emptier the
   off-diagonal cells.

In [ ]:
def icc1(x, g):
    """One-way ICC: between-video variance as a share of the total (0 = video says nothing)."""
    d = pd.DataFrame({"x": np.asarray(x, float), "g": np.asarray(g)})
    k, N = d.g.nunique(), len(d)
    if k < 2:
        return float("nan")
    gm = d.groupby("g")["x"].agg(["mean", "count"])
    msb = (gm["count"] * (gm["mean"] - d.x.mean()) ** 2).sum() / (k - 1)
    msw = sum(((s.x - s.x.mean()) ** 2).sum() for _, s in d.groupby("g")) / (N - k)
    n0 = (N - (gm["count"] ** 2).sum() / N) / (k - 1)
    return float((msb - msw) / (msb + (n0 - 1) * msw))


def bimodal_bic(x):
    """BIC of a 2-component GMM minus a 1-component one: negative = two humps."""
    from sklearn.mixture import GaussianMixture
    v = np.asarray(x, float).reshape(-1, 1)
    return (GaussianMixture(2, random_state=0).fit(v).bic(v)
            - GaussianMixture(1, random_state=0).fit(v).bic(v))


for c in ("frame_similarity", "gaze_patch_token_sim"):
    icc, dbic = icc1(train_pool[c], train_pool["sequence"]), bimodal_bic(train_pool[c])
    print(f"{c}")
    print(f"   ICC (scene identity share) : {icc:+.3f}   {'<- HIGH: consider PER_VIDEO = True' if icc > 0.4 else 'ok'}")
    print(f"   BIC(2) - BIC(1)            : {dbic:+,.0f}   "
          f"{'bimodal -- a data-driven cut is meaningful' if dbic < 0 else 'unimodal -- median is honest'}")
    print(f"   median                     : {train_pool[c].median():.4f}")

fig, ax = plt.subplots(1, 3, figsize=(17, 4.4))
for a, c, col in zip(ax[:2], ("frame_similarity", "gaze_patch_token_sim"), ("tab:blue", "tab:orange")):
    a.hist(train_pool[c], bins=60, color=col, alpha=.8)
    a.axvline(train_pool[c].median(), c="k", ls="--", lw=1.2, label=f"median {train_pool[c].median():.3f}")
    a.set_xlabel(c); a.set_ylabel("rows"); a.legend()
ax[2].scatter(lite.frame_similarity, lite.gaze_patch_token_sim, s=2, alpha=.08)
ax[2].axvline(train_pool.frame_similarity.median(), c="k", lw=.9, ls="--")
ax[2].axhline(train_pool.gaze_patch_token_sim.median(), c="k", lw=.9, ls="--")
ax[2].set_xlabel("frame_similarity"); ax[2].set_ylabel("gaze_patch_token_sim")
r = float(np.corrcoef(lite.frame_similarity, lite.gaze_patch_token_sim)[0, 1])
ax[2].set_title(f"joint distribution, r = {r:+.3f}")
plt.tight_layout(); plt.show()

NAMES = {0: "TRANSITION", 1: "PURSUIT", 2: "REFIXATION", 3: "STABLE"}

def shares(tf, tg, d=None):
    d = lite if d is None else d
    q = 2 * (d.frame_similarity.to_numpy() > tf).astype(int) + (d.gaze_patch_token_sim.to_numpy() > tg).astype(int)
    return {NAMES[i]: 100 * float((q == i).mean()) for i in range(4)}

rows = []
for fp in (40, 50, 60):
    for gp in (40, 50, 60):
        tf = np.percentile(train_pool.frame_similarity, fp)
        tg = np.percentile(train_pool.gaze_patch_token_sim, gp)
        s = shares(tf, tg)
        rows.append(dict(frame_pct=fp, gaze_pct=gp, tau_frame=round(tf, 4), tau_gaze=round(tg, 4),
                         **{k: round(v, 1) for k, v in s.items()}, min_cell=round(min(s.values()), 1)))
print("\nclass shares (%) at neighbouring percentile cuts -- if they swing wildly, the labels are fragile:")
display(pd.DataFrame(rows))

## 8 — Choose the thresholds, label, write the folds

τ is computed **once, on the pooled non-test rows**, and used for every fold, so labels mean the
same thing in every fold. The cell prints how far each fold's own threshold would have drifted.

Reasons to move off the medians (section 7):
- **`REFIXATION` under ~5%** — it is the class that justifies two thresholds. To grow it, **lower**
  `FRAME_PCT` and **raise** `GAZE_PCT`.
- **High ICC** — `PER_VIDEO = True` takes percentiles within each video. That changes what a label
  means ("more dissimilar than usual *for this scene*"), which a fixed-threshold gate cannot do at
  deployment. Use it knowingly.

In [ ]:
FRAME_PCT = 50        # percentile for tau_frame
GAZE_PCT  = 50        # percentile for tau_gaze
PER_VIDEO = False     # True -> percentiles within each video (see above)
MIN_CELL  = 5.0       # warn if any class falls below this share (%)

TAU_F = float(np.percentile(train_pool.frame_similarity, FRAME_PCT))
TAU_G = float(np.percentile(train_pool.gaze_patch_token_sim, GAZE_PCT))
if not PER_VIDEO:
    print(f"GLOBAL thresholds from {len(train_pool):,} pooled rows")
    print(f"   tau_frame = {TAU_F:.4f}   (p{FRAME_PCT})")
    print(f"   tau_gaze  = {TAU_G:.4f}   (p{GAZE_PCT})")
    tf_row, tg_row = np.full(len(lite), TAU_F), np.full(len(lite), TAU_G)
    print("\n   had each fold computed its own thresholds from its training videos:")
    sp = []
    for k in range(K):
        tr = train_pool[train_pool.fold != k]
        a, b = np.percentile(tr.frame_similarity, FRAME_PCT), np.percentile(tr.gaze_patch_token_sim, GAZE_PCT)
        sp.append((a, b))
        print(f"      fold {k}: tau_frame {a:.4f} ({a - TAU_F:+.4f})   tau_gaze {b:.4f} ({b - TAU_G:+.4f})")
    drift = float(np.abs(np.array(sp) - [TAU_F, TAU_G]).max())
    print(f"   max drift {drift:.4f}  -> {'negligible' if drift < 0.01 else 'LARGE: consider per-fold thresholds'}")
else:
    tf_map = train_pool.groupby("sequence").frame_similarity.quantile(FRAME_PCT / 100)
    tg_map = train_pool.groupby("sequence").gaze_patch_token_sim.quantile(GAZE_PCT / 100)
    tf_row = lite.sequence.map(tf_map).fillna(TAU_F).to_numpy()      # test videos: the pooled value
    tg_row = lite.sequence.map(tg_map).fillna(TAU_G).to_numpy()
    print(f"PER-VIDEO thresholds for {len(tf_map)} videos; test videos use the pooled value")

quad = 2 * (lite.frame_similarity.to_numpy() > tf_row).astype(int) + (lite.gaze_patch_token_sim.to_numpy() > tg_row).astype(int)
lite["quad"] = quad
counts = pd.Series(quad).value_counts().reindex(range(4), fill_value=0)
print()
for i in range(4):
    print(f"   {i}  {NAMES[i]:<12} {counts[i]:>7,}  {100 * counts[i] / len(lite):5.1f}%   {'SKIP' if i == 3 else 'SEND'}")
low = [NAMES[i] for i in range(4) if 100 * counts[i] / len(lite) < MIN_CELL]
if low:
    print(f"\n!! below {MIN_CELL}%: {low}" + ("  -- lower FRAME_PCT / raise GAZE_PCT to grow REFIXATION"
                                            if "REFIXATION" in low else ""))

grp = np.where(lite.is_test == 1, "test", "fold " + lite.fold.astype(str))
bal = (pd.crosstab(grp, lite["quad"], normalize="index") * 100).reindex(columns=range(4), fill_value=0) \
        .rename(columns=NAMES).round(1)
print("\nclass balance (%) per fold and test:")
display(bal)

# ---- write the labelled master, the test file and the 5 fold files ----------------------
full = pd.read_csv(CSV)
assert len(full) == len(lite) and (full["sequence"].to_numpy() == lite["sequence"].to_numpy()).all()
full["frame_high"], full["gaze_high"] = (quad >> 1).astype(int), (quad & 1).astype(int)
full["quad"], full["quad_label"] = quad, pd.Series(quad).map(NAMES).to_numpy()
full["gate"] = np.where(quad == 3, "DISCARD", "SEND")
full["fold"], full["is_test"] = lite["fold"].to_numpy(), lite["is_test"].to_numpy()
full.to_csv(CSV_LAB, index=False)

test_df = full[full.is_test == 1].copy()
test_df["idx"] = range(len(test_df))
test_df.to_csv(os.path.join(FOLD_DIR, "test.csv"), index=False)
for k in range(K):
    d = full[full.is_test == 0].copy()
    d["split"] = np.where(d["fold"] == k, "val", "train")
    d["idx"] = range(len(d))
    d.to_csv(os.path.join(FOLD_DIR, f"train_val_fold{k+1}.csv"), index=False)

json.dump(dict(tau_frame=TAU_F, tau_gaze=TAU_G, frame_pct=FRAME_PCT, gaze_pct=GAZE_PCT,
               per_video=PER_VIDEO, dead_zone=0.0, k_folds=K, test_every=TEST_EVERY,
               computed_on="pooled non-test rows", n_pool_rows=int(len(train_pool)), names=NAMES,
               test_videos=sorted(test_videos), fold_of=fold_of, source=os.path.basename(CSV)),
          open(THRESH, "w"), indent=2)

# ---- verify what was written -------------------------------------------------------------
te = pd.read_csv(os.path.join(FOLD_DIR, "test.csv"), usecols=["sequence"])
fds = [pd.read_csv(os.path.join(FOLD_DIR, f"train_val_fold{k+1}.csv"), usecols=["sequence", "split"]) for k in range(K)]
val_sets = [set(d[d.split == "val"].sequence) for d in fds]
pool_set = set(fds[0].sequence)
for name, v in [("row counts add up", len(te) + len(fds[0]) == len(lite)),
                ("no test video in any fold", not (set(te.sequence) & pool_set)),
                ("each training video validated exactly once",
                 set().union(*val_sets) == pool_set and sum(map(len, val_sets)) == len(pool_set)),
                ("train and val never share a video",
                 all(not (set(d[d.split == 'train'].sequence) & set(d[d.split == 'val'].sequence)) for d in fds))]:
    print(f"   [{'PASS' if v else 'FAIL'}]  {name}")
print(f"\nwrote {FOLD_DIR}/ (5 folds + test) and {THRESH}")
print(f"on test, a perfect gate would skip {100 * (quad[lite.is_test == 1] == 3).mean():.1f}% of frames")

## 9 — Train: `L = L_NCE + L_sim`

Two losses, both pulling the gaze-only student toward the pixel teacher:

| loss | what it does |
|---|---|
| `L_sim` | MSE regression of the two teacher scores `[S_frame, S_gaze]` from the student's embedding `z_M` |
| `L_NCE` | InfoNCE (τ = 0.1): `z_M` must match `ẑ_M`, an embedding predicted by `h` from the two frozen DINOv2 features, against the other rows of the batch |

**`L_NCE`'s negatives come from the batch**, so a batch holds at most `--max_per_video` (8) rows of
any one video: two windows seconds apart in the same video are near-duplicates and would be false
negatives. With few training videos this caps the effective batch size, printed below.

The first run collapses the frame features into one cached matrix (`feat_cache.npz`); later runs
reuse it. Watch **both** loss terms: if `sim` stops falling while `nce` keeps improving, the
contrastive term is dominating, so lower `--nce_weight`.

In [ ]:
FOLDS = FOLD_DIR
RUNS  = os.path.join(RUNS_DIR, "joint")
th = json.load(open(THRESH))
TAU_F, TAU_G = th["tau_frame"], th["tau_gaze"]
NAMES = {0: "TRANSITION", 1: "PURSUIT", 2: "REFIXATION", 3: "STABLE"}

# the folds must reproduce their labels from the stored thresholds
for f in [f"train_val_fold{k}.csv" for k in range(1, 6)] + ["test.csv"]:
    d = pd.read_csv(os.path.join(FOLDS, f), usecols=["sequence", "frame_similarity", "gaze_patch_token_sim", "quad"])
    q = 2 * (d.frame_similarity > TAU_F).astype(int) + (d.gaze_patch_token_sim > TAU_G).astype(int)
    agree = float((q == d["quad"]).mean())
    print(f"   {f:<24} {len(d):6,} rows  {d.sequence.nunique():3d} videos   "
          f"labels reproduced on {100 * agree:6.2f}%   [{'PASS' if agree > 0.999 or th['per_video'] else 'FAIL'}]")

EPOCHS, PATIENCE, BATCH, LR = 200, 25, 512, 3e-4
MAX_PER_VIDEO = 8
RATES = "gaze_rates_window"        # or gaze_vec3d_rates_window (exact speed)

f1 = pd.read_csv(os.path.join(FOLDS, "train_val_fold1.csv"), usecols=["sequence", "split"])
EFF_BATCH = min(BATCH, MAX_PER_VIDEO * f1[f1.split == "train"].sequence.nunique())
print(f"\neffective InfoNCE batch: about {EFF_BATCH} rows  (ln = {np.log(EFF_BATCH):.2f}, roughly where L_NCE starts)")

t0 = time.time()
!python -u -m gatetrain.train \
    --folds_dir "{FOLDS}" --out_dir "{RUNS}" --thresholds "{THRESH}" \
    --rates_col {RATES} --epochs {EPOCHS} --patience {PATIENCE} \
    --batch_size {BATCH} --lr {LR} --max_per_video {MAX_PER_VIDEO} \
    --feat_cache "{FEAT_CACHE}" --feat_root "{FEAT_DIR}"
print(f"\ntotal {(time.time() - t0) / 60:.1f} min")

## 10 — Curves and the fold spread

All CV figures are the **mean over the last 5 epochs**, never the best epoch: a maximum over a
noisy curve is biased upward even when there is no signal. **Read R² first** — it has a baseline
(0 = predicting the training mean); AUC can look respectable when R² is not.

In [ ]:
res = json.load(open(os.path.join(RUNS, "cv_results.json")))
H = res["history"]

fig, ax = plt.subplots(1, 4, figsize=(20, 4.2))
for k, h in H.items():
    ep = [r["epoch"] for r in h]
    ax[0].plot(ep, [r["tr_sim"] for r in h], lw=1.2, label=k)
    if "tr_nce" in h[0]:
        ax[1].plot(ep, [r.get("tr_nce", np.nan) for r in h], lw=1.2, label=k)
    ax[2].plot(ep, [r["auc_gate"] for r in h], lw=1.4, label=k)
    ax[3].plot(ep, [r["r2_mean"] for r in h], lw=1.4, label=k)
ax[0].set_title("train L_sim"); ax[0].set_xlabel("epoch"); ax[0].legend(fontsize=7)
ax[1].set_title("train L_NCE"); ax[1].set_xlabel("epoch")
ax[1].axhline(np.log(EFF_BATCH), ls="--", c="k", lw=1, label="ln(batch)"); ax[1].legend(fontsize=7)
ax[2].axhline(0.5, ls="--", c="k", lw=1); ax[2].set_title("val AUC_gate", fontweight="bold"); ax[2].set_xlabel("epoch")
ax[3].axhline(0.0, ls="--", c="k", lw=1, label="predict-the-mean")
ax[3].set_title("val R2", fontweight="bold"); ax[3].set_xlabel("epoch"); ax[3].legend(fontsize=7)
plt.tight_layout(); plt.show()

tab = pd.DataFrame([f["last"] for f in res["folds"]])
tab.index = [f"fold {i+1}" for i in range(len(tab))]
display(tab[["auc_gate", "auc_frame", "auc_gaze", "r2_mean", "quad_acc", "recall_q2"]].round(4))

auc = tab.auc_gate.to_numpy(); r2 = tab.r2_mean.to_numpy()
print(f"AUC_gate  {auc.mean():.4f} +/- {auc.std():.4f}")
print(f"R2        {r2.mean():+.3f} +/- {r2.std():.3f}   <- read this first; it has a baseline")
print(f"REFIX rec {tab.recall_q2.mean():.3f}")
if r2.mean() <= 0:
    print("\n  R2 <= 0: no better than predicting the training mean on held-out videos.")

## 11 — Ablation: is `L_NCE` earning its place?

`--sim_only` trains `L_sim` alone: same data, same head, no contrastive term and no frame features.
If the joint run is not better than this, the contrastive machinery (the change predictor `h` and
loading every frame's features) costs complexity for nothing.

In [ ]:
RUNS_SIM = os.path.join(RUNS_DIR, "sim_only")
!python -u -m gatetrain.train \
    --folds_dir "{FOLDS}" --out_dir "{RUNS_SIM}" --thresholds "{THRESH}" \
    --rates_col {RATES} --epochs {EPOCHS} --patience {PATIENCE} \
    --batch_size {BATCH} --lr {LR} --max_per_video {MAX_PER_VIDEO} --sim_only 2>&1 | tail -14

sim = json.load(open(os.path.join(RUNS_SIM, "cv_results.json")))
j_auc = np.array([f["last"]["auc_gate"] for f in res["folds"]])
s_auc = np.array([f["last"]["auc_gate"] for f in sim["folds"]])
j_r2  = np.array([f["last"]["r2_mean"] for f in res["folds"]])
s_r2  = np.array([f["last"]["r2_mean"] for f in sim["folds"]])

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for a_, (j, s, t) in zip(ax, [(j_auc, s_auc, "AUC_gate"), (j_r2, s_r2, "R2")]):
    a_.bar(np.arange(5) - .19, j, .38, label=f"joint  {j.mean():.3f}")
    a_.bar(np.arange(5) + .19, s, .38, label=f"sim_only  {s.mean():.3f}", color="tab:orange")
    a_.set_xticks(range(5)); a_.set_xticklabels([f"f{i+1}" for i in range(5)]); a_.set_title(t); a_.legend()
ax[0].axhline(0.5, ls="--", c="k", lw=1); ax[1].axhline(0, ls="--", c="k", lw=1)
plt.tight_layout(); plt.show()

d = j_auc.mean() - s_auc.mean()
print(f"joint - sim_only:  AUC {d:+.4f}   R2 {j_r2.mean() - s_r2.mean():+.4f}   "
      f"(fold spread of the joint R2: {j_r2.std():.3f})")
print("  A difference smaller than the fold spread is not evidence either way.")

## 12 — Shuffle control

Each row's frames are paired with **another row's** gaze, destroying the correspondence both losses
depend on. **R² must stay ≤ 0 and AUC near 0.5.** If not, the setup leaks and nothing above is
credible.

In [ ]:
RUNS_CTRL = os.path.join(RUNS_DIR, "joint_control")
!python -u -m gatetrain.train \
    --folds_dir "{FOLDS}" --out_dir "{RUNS_CTRL}" --thresholds "{THRESH}" \
    --rates_col {RATES} --epochs {EPOCHS} --patience {PATIENCE} \
    --batch_size {BATCH} --lr {LR} --max_per_video {MAX_PER_VIDEO} \
    --feat_cache "{FEAT_CACHE}" --feat_root "{FEAT_DIR}" --shuffle_control 2>&1 | tail -12

ctrl = json.load(open(os.path.join(RUNS_CTRL, "cv_results.json")))
c_auc = np.array([f["last"]["auc_gate"] for f in ctrl["folds"]])
c_r2  = np.array([f["last"]["r2_mean"] for f in ctrl["folds"]])

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].bar(np.arange(5) - .19, j_auc, .38, label=f"real  {j_auc.mean():.3f}")
ax[0].bar(np.arange(5) + .19, c_auc, .38, label=f"shuffled  {c_auc.mean():.3f}", color="tab:red")
ax[0].axhline(0.5, ls="--", c="k", lw=1); ax[0].set_title("AUC_gate"); ax[0].legend()
ax[1].bar(np.arange(5) - .19, j_r2, .38, label=f"real  {j_r2.mean():+.3f}")
ax[1].bar(np.arange(5) + .19, c_r2, .38, label=f"shuffled  {c_r2.mean():+.3f}", color="tab:red")
ax[1].axhline(0, ls="--", c="k", lw=1); ax[1].set_title("R2"); ax[1].legend()
for a_ in ax:
    a_.set_xticks(range(5)); a_.set_xticklabels([f"f{i+1}" for i in range(5)])
plt.tight_layout(); plt.show()

print(f"real     AUC {j_auc.mean():.4f}   R2 {j_r2.mean():+.3f}")
print(f"shuffled AUC {c_auc.mean():.4f}   R2 {c_r2.mean():+.3f}")
if c_r2.mean() > 0.05:
    print("\n  !! THE CONTROL LEARNED SOMETHING. Suspect the split or the normalisation.")
elif j_r2.mean() - c_r2.mean() < 0.05:
    print("\n  Real and shuffled are indistinguishable: gaze is not predicting these scores at this scale.")
else:
    print("\n  The control fails as it must, and the real run is clearly above it.")

## 13 — Test, once, through the SEND / SKIP rule

The five fold models are averaged, their predictions converted back to raw cosine units, and the
decision rule applied at the τ that made the labels:

```
S_frame > tau_frame  AND  S_gaze > tau_gaze   ->  SKIP
otherwise                                     ->  SEND
```

Read the per-class table: it is where `REFIXATION` — the case two thresholds exist for — is
either recovered or not.

In [ ]:
PRED = os.path.join(RUNS, "test_preds.csv")
!python -u -m gatetrain.infer \
    --folds_dir "{FOLDS}" --ckpt_dir "{RUNS}" --thresholds "{THRESH}" --out_csv "{PRED}"

In [ ]:
from src.foldtrain.metrics import roc_auc

pr = pd.read_csv(PRED)
fig, ax = plt.subplots(1, 3, figsize=(17, 4.6))
for k, (pc, tc, tau) in enumerate([("pred_S_frame", "frame_similarity", TAU_F),
                                   ("pred_S_gaze", "gaze_patch_token_sim", TAU_G)]):
    ax[k].scatter(pr[tc], pr[pc], s=4, alpha=.15)
    lo, hi = min(pr[tc].min(), pr[pc].min()), max(pr[tc].max(), pr[pc].max())
    ax[k].plot([lo, hi], [lo, hi], "r--", lw=1, label="perfect")
    ax[k].axvline(tau, c="k", lw=.8, ls=":"); ax[k].axhline(tau, c="k", lw=.8, ls=":")
    ax[k].set_xlabel(f"true {tc}"); ax[k].set_ylabel("predicted")
    ax[k].set_title(f"{tc}\nr = {np.corrcoef(pr[tc], pr[pc])[0, 1]:+.3f}   (dotted = tau)"); ax[k].legend(fontsize=8)

cm = pd.crosstab(pr.true_quad, pr.pred_quad).reindex(index=list(NAMES.values()), columns=list(NAMES.values()), fill_value=0)
im = ax[2].imshow(cm.to_numpy(), cmap="Blues")
ax[2].set_xticks(range(4)); ax[2].set_xticklabels(NAMES.values(), rotation=30, ha="right")
ax[2].set_yticks(range(4)); ax[2].set_yticklabels(NAMES.values())
for i in range(4):
    for j in range(4):
        v = cm.to_numpy()[i, j]
        ax[2].text(j, i, f"{v:,}", ha="center", va="center", fontsize=8,
                   color="white" if v > cm.to_numpy().max() / 2 else "black")
ax[2].set_xlabel("predicted"); ax[2].set_ylabel("truth"); ax[2].set_title("classes")
plt.tight_layout(); plt.show()

print("Points in the off-diagonal quadrants of the left two panels flip the decision:")
print("the model put a row on the wrong side of tau.")
print(f"\ntest accuracy on the 4 classes: {pr.correct.mean():.4f}")
display(pr.reason.value_counts().rename("rows").to_frame().assign(pct=lambda d: (100 * d.rows / len(pr)).round(1)))

## 14 — Reading it

| | |
|---|---|
| **R²** | read first. ≤ 0 means no better than predicting the mean, whatever AUC says |
| **AUC_gate** | ranking quality on `min(Ŝ_frame − τ_f, Ŝ_gaze − τ_g)`, the margin on the weaker axis |
| **REFIXATION recall** | the class that justifies two thresholds. Near zero is a finding, not a detail |
| **FALSE SKIP** | content lost for good. Not symmetric with FALSE SEND |
| **fold spread** | wide → the headline mostly reflects which videos landed in validation |
| **control** | must fail. If it does not, fix the leak first |

**If R² is at or below zero:** try `RATES = "gaze_vec3d_rates_window"` (exact speed); try
`--nce_weight 0.1`; compare with `--sim_only`; check the shuffle control landed at zero.

**What is not tested here:** whether these thresholds match anything a **VLM** would notice. The
labels are DINOv2 cosines cut at percentiles we chose; every number here measures agreement with
that proxy.

## 15 — Export the trained gate

`/content` is wiped when the runtime ends. This writes a **self-contained release** (to Drive too,
with `SAVE_TO_DRIVE`) that runs without this notebook, without the repo and without retraining.

| file | |
|---|---|
| `gate_ensemble.ts.pt` | the 5 folds averaged (TorchScript) — what the test numbers above come from |
| `gate_best_fold.ts.pt` | the single fold with the best validation score, one small model |
| `gate_weights.pt` | plain `state_dict`s + statistics, for use inside the repo |
| `model_card.json` | thresholds, normalisation, architecture, metrics |
| `example_inference.py`, `README.md` | usage, next to the files |

**The exported model is not the trained model.** `h`, the change predictor, exists only to build
the `L_NCE` target and most of the parameters are in it; it is dropped. What ships is `f_M` + head.
**The export absorbs the normalisation**: it takes raw gaze rates and returns raw cosine
similarities, directly comparable to τ.

In [ ]:
import torch.nn as nn
from gatetrain.model import build

RELEASE = os.path.join(WORK, "release", "gate_v1")
os.makedirs(RELEASE, exist_ok=True)

cks = [torch.load(os.path.join(RUNS, f"fold{k}.pt"), map_location="cpu") for k in range(1, 6)]
ref = cks[0]["args"]
best_i = int(np.argmax([c["best"] for c in cks]))
print(f"per-fold best val {ref['select_on']}: " + "  ".join(f"f{i+1} {c['best']:.4f}" for i, c in enumerate(cks)))
print(f"best single fold : fold{best_i+1}")


class DeployableGate(nn.Module):
    """Raw gaze rates in, raw cosine similarities out (the training model works on standardised
    rates and targets; both transforms are folded in here). `h` is not carried."""

    def __init__(self, ck):
        super().__init__()
        m = build(feat_dim=ck.get("feat_dim", 384), dim=ref["dim"], hidden=ref["hidden"],
                  width=ref["width"], fusion=ref["fusion"], dropout=0.0)
        m.load_state_dict(ck["model"])
        self.f_m, self.head = m.f_m, m.head
        rm, rs = (torch.tensor(v, dtype=torch.float32) for v in ck["rate_stats"])
        tm, ts = (torch.tensor(v, dtype=torch.float32) for v in ck["target_stats"])
        self.register_buffer("rm", rm.view(1, -1, 1)); self.register_buffer("rs", rs.view(1, -1, 1))
        self.register_buffer("tm", tm); self.register_buffer("ts", ts)

    def forward(self, rates, mask):
        """rates (B,3,L) raw rad/s, mask (B,L) bool -> (B,2) raw [S_frame, S_gaze]."""
        return self.head(self.f_m((rates - self.rm) / self.rs, mask)) * self.ts + self.tm


class EnsembleGate(nn.Module):
    """Mean of the folds in RAW units: each fold un-standardises with its own statistics first."""

    def __init__(self, models):
        super().__init__()
        self.models = nn.ModuleList(models)

    def forward(self, rates, mask):
        return torch.stack([m(rates, mask) for m in self.models], 0).mean(0)


L = ref["seq_len"]
ex_r, ex_m = torch.randn(4, 3, L), torch.ones(4, L, dtype=torch.bool)
single = DeployableGate(cks[best_i]).eval()
ens = EnsembleGate([DeployableGate(c) for c in cks]).eval()

# torch.jit.trace records one concrete execution, so check it generalises and honours the mask
chk_r = torch.randn(3, 3, L)
chk_m = torch.ones(3, L, dtype=torch.bool); chk_m[1, L // 2:] = False
with torch.no_grad():
    for name, mod in (("gate_best_fold", single), ("gate_ensemble", ens)):
        ts_mod = torch.jit.trace(mod, (ex_r, ex_m))
        assert ts_mod(torch.randn(17, 3, L), torch.ones(17, L, dtype=torch.bool)).shape == (17, 2)
        assert torch.allclose(ts_mod(chk_r, chk_m), mod(chk_r, chk_m), atol=1e-6), f"{name}: trace disagrees"
        assert not torch.allclose(ts_mod(chk_r, chk_m), ts_mod(chk_r, torch.ones_like(chk_m))), f"{name}: mask ignored"
        torch.jit.save(ts_mod, os.path.join(RELEASE, f"{name}.ts.pt"))
        print(f"   {name:<16} traced, generalises to other batch sizes, mask honoured")

n_dep = sum(p.numel() for p in single.parameters())
n_train = sum(p.numel() for p in build().parameters())
print(f"\ndeployable parameters: {n_dep:,}  (training model: {n_train:,})")

torch.save(dict(folds=[dict(model=c["model"], rate_stats=c["rate_stats"], target_stats=c["target_stats"],
                            best_val=float(c["best"]), fold=c["fold"]) for c in cks],
                best_fold_index=best_i, arch={k: ref[k] for k in ("dim", "hidden", "width", "fusion")},
                rates_col=ref["rates_col"], seq_len=L, tau_frame=TAU_F, tau_gaze=TAU_G),
           os.path.join(RELEASE, "gate_weights.pt"))

n_test_videos = pd.read_csv(os.path.join(FOLDS, "test.csv"), usecols=["sequence"]).sequence.nunique()
refix = pr.true_quad == "REFIXATION"
refix_recall = float((refix & (pr.pred_quad == "REFIXATION")).sum() / max(1, refix.sum()))
card = dict(
    name="gaze-gate", created=time.strftime("%Y-%m-%d %H:%M:%S"),
    what_it_does="Predicts [S_frame, S_gaze] (DINOv2 cosine similarity between consecutive frames, whole "
                 "frame and at the gaze point) from a window of gaze angular velocities alone. "
                 "No frame is encoded at inference.",
    input=dict(rates=f"(B, 3, {L}) float32, raw rad/s, channels [omega_yaw, omega_pitch, omega_mag]",
               mask=f"(B, {L}) bool, True where the step is real", seq_len=L, n_channels=3,
               source_column=ref["rates_col"]),
    output=dict(shape="(B, 2) float32", meaning=["S_frame - whole-frame cosine similarity",
                                                 "S_gaze  - gaze-region patch-token cosine similarity"],
                units="raw cosine, directly comparable to the thresholds below"),
    thresholds=dict(tau_frame=TAU_F, tau_gaze=TAU_G,
                    rule="SKIP the frame iff S_frame > tau_frame AND S_gaze > tau_gaze; otherwise SEND it",
                    provenance=f"percentiles {th['frame_pct']} / {th['gaze_pct']} of each score over the pooled non-test rows"),
    quadrants={"0": "TRANSITION (both low)", "1": "PURSUIT (frame low, gaze high)",
               "2": "REFIXATION (frame high, gaze low)", "3": "STABLE (both high) = SKIP"},
    architecture={k: ref[k] for k in ("dim", "hidden", "width", "fusion")},
    parameters=dict(deployed=int(n_dep), training_total=int(n_train), dropped="h, the change predictor"),
    training=dict(objective=f"L_sim + {ref['nce_weight']:g} * L_NCE", epochs_max=ref["epochs"],
                  batch_size=ref["batch_size"], lr=ref["lr"], folds=5, split="by video",
                  selected_on=ref["select_on"], videos=int(data["sequence"].nunique())),
    metrics=dict(
        headline_test=dict(source=f"gatetrain.infer, 5-model ensemble, {len(pr)} rows / {n_test_videos} videos",
                           auc_gate=None, refixation_recall=refix_recall, note="filled by the next cell"),
        cv_last5=dict(auc_gate_mean=float(auc.mean()), auc_gate_std=float(auc.std()), r2_mean=float(r2.mean()),
                      r2_std=float(r2.std())),
        sim_only_ablation=dict(auc_gate=float(s_auc.mean()), r2=float(s_r2.mean())),
        shuffle_control=dict(auc_gate=float(c_auc.mean()), r2=float(c_r2.mean()), note="must be near 0.5 / <= 0"),
    ),
    caveats=["Thresholds are percentiles, not calibrated against any VLM; no VLM was run on gated vs ungated streams.",
             "Selected on validation; quote the test numbers, not the best-fold validation score."],
)
json.dump(card, open(os.path.join(RELEASE, "model_card.json"), "w"), indent=2)
shutil.copy(os.path.join(RUNS, "cv_results.json"), RELEASE)
shutil.copy(PRED, os.path.join(RELEASE, "test_preds.csv"))
shutil.copy(THRESH, os.path.join(RELEASE, "thresholds.json"))
print(f"\nwrote -> {RELEASE}")

### Verify the bundle before trusting it

Reloads the TorchScript, builds the test input **without importing anything from the repo's
training code**, and checks the predictions match `gatetrain.infer` to floating-point tolerance.

In [ ]:
card = json.load(open(os.path.join(RELEASE, "model_card.json")))
gate = torch.jit.load(os.path.join(RELEASE, "gate_ensemble.ts.pt")).eval()
TF, TG = card["thresholds"]["tau_frame"], card["thresholds"]["tau_gaze"]
SEQ, RATES_COL = int(card["input"]["seq_len"]), card["input"]["source_column"]


def pack(series, seq_len):
    """'a,b,c;a,b,c;...' -> (N,3,L) raw rates + (N,L) mask, re-implemented here on purpose."""
    X = np.zeros((len(series), seq_len, 3), np.float32)
    M = np.zeros((len(series), seq_len), bool)
    for i, s in enumerate(series):
        if not isinstance(s, str) or not s.strip():
            continue
        r = np.array([[float(v) for v in t.split(",")] for t in s.split(";")], np.float32)
        k = min(len(r), seq_len)
        X[i, :k], M[i, :k] = r[:k], True
    return np.ascontiguousarray(X.transpose(0, 2, 1)), M


te = pd.read_csv(os.path.join(FOLDS, "test.csv"))
X, M = pack(te[RATES_COL].tolist(), SEQ)
with torch.no_grad():
    pred = gate(torch.from_numpy(X), torch.from_numpy(M)).numpy()

ref_csv = pd.read_csv(os.path.join(RELEASE, "test_preds.csv"))
diff = max(np.abs(pred[:, 0] - ref_csv.pred_S_frame.to_numpy()).max(),
           np.abs(pred[:, 1] - ref_csv.pred_S_gaze.to_numpy()).max())
ok = diff < 1e-4
print(f"max |bundle - gatetrain.infer| = {diff:.2e}   [{'PASS' if ok else 'FAIL'}]")

skip = (pred[:, 0] > TF) & (pred[:, 1] > TG)
truth_skip = te["quad"].to_numpy() == 3
gate_auc = roc_auc(truth_skip, np.minimum(pred[:, 0] - TF, pred[:, 1] - TG))
print(f"\nfrom the bundle alone, on the test videos:")
print(f"   AUC_gate        {gate_auc:.4f}")
print(f"   frames skipped  {100 * skip.mean():.1f}%")
print(f"   FALSE SKIP      {100 * float((skip & ~truth_skip).mean()):.1f}%   content lost")
print(f"   agreement       {100 * float((skip == truth_skip).mean()):.1f}%")
print(f"   REFIX recall    {refix_recall:.3f}")

card["metrics"]["headline_test"].update(auc_gate=float(gate_auc), frames_skipped_pct=float(100 * skip.mean()),
                                        false_skip_pct=float(100 * (skip & ~truth_skip).mean()),
                                        note="recomputed from the exported bundle")
card["verified"] = dict(passed=bool(ok), max_abs_diff_vs_infer=float(diff))
json.dump(card, open(os.path.join(RELEASE, "model_card.json"), "w"), indent=2)
assert ok, "the bundle does NOT reproduce the training run"

### Usage docs next to the weights

In [ ]:
EXAMPLE = r'''"""Run the gaze gate. Requires torch and numpy only.

    python example_inference.py              # demo on random input
    python example_inference.py rows.csv     # a CSV with a gaze-rates column
"""
import json, os, sys
import numpy as np
import torch

HERE = os.path.dirname(os.path.abspath(__file__))
CARD = json.load(open(os.path.join(HERE, "model_card.json")))
TAU_F, TAU_G = CARD["thresholds"]["tau_frame"], CARD["thresholds"]["tau_gaze"]
SEQ, QUAD = int(CARD["input"]["seq_len"]), CARD["quadrants"]
gate = torch.jit.load(os.path.join(HERE, "gate_ensemble.ts.pt")).eval()


def pack(series, seq_len=SEQ):
    """'wy,wp,wm;wy,wp,wm;...' x N -> (N,3,L) raw rates, (N,L) validity mask."""
    X = np.zeros((len(series), seq_len, 3), np.float32)
    M = np.zeros((len(series), seq_len), bool)
    for i, s in enumerate(series):
        if not isinstance(s, str) or not s.strip():
            continue
        r = np.array([[float(v) for v in t.split(",")] for t in s.split(";")], np.float32)
        k = min(len(r), seq_len)
        X[i, :k], M[i, :k] = r[:k], True
    return np.ascontiguousarray(X.transpose(0, 2, 1)), M


def predict(rates, mask):
    """(B,3,L) raw rad/s + (B,L) bool -> (B,2) raw cosine [S_frame, S_gaze]."""
    with torch.no_grad():
        return gate(torch.as_tensor(rates, dtype=torch.float32), torch.as_tensor(mask, dtype=torch.bool)).numpy()


def should_skip(pred):
    """Skip only when BOTH scores clear their threshold."""
    return (pred[:, 0] > TAU_F) & (pred[:, 1] > TAU_G)


if __name__ == "__main__":
    if len(sys.argv) > 1:
        import pandas as pd
        df = pd.read_csv(sys.argv[1])
        X, M = pack(df[CARD["input"]["source_column"]].tolist())
    else:
        print("no CSV given -- random input, numbers are meaningless\n")
        X, M = np.random.randn(6, 3, SEQ).astype(np.float32) * 0.5, np.ones((6, SEQ), bool)
    p = predict(X, M)
    skip = should_skip(p)
    q = 2 * (p[:, 0] > TAU_F).astype(int) + (p[:, 1] > TAU_G).astype(int)
    for i in range(min(len(p), 20)):
        print(f"{i:>5} {p[i,0]:>9.4f} {p[i,1]:>9.4f}  {'SKIP' if skip[i] else 'SEND':<5} {QUAD[str(q[i])]}")
    print(f"\n{100*skip.mean():.1f}% of {len(p)} rows would be skipped")
'''

m = card["metrics"]
README = f"""# gaze gate

Decides whether a video frame is worth sending to a vision-language model from the wearer's gaze
velocity alone. No frame is encoded at inference.

```python
import torch, json
gate = torch.jit.load("gate_ensemble.ts.pt").eval()
card = json.load(open("model_card.json"))
tf, tg = card["thresholds"]["tau_frame"], card["thresholds"]["tau_gaze"]
# rates: (B, 3, {L}) float32, RAW rad/s [omega_yaw, omega_pitch, omega_mag]; mask: (B, {L}) bool
pred = gate(rates, mask)                        # (B, 2) raw cosine [S_frame, S_gaze]
skip = (pred[:, 0] > tf) & (pred[:, 1] > tg)    # skip only if BOTH clear
```

Or run `python example_inference.py rows.csv`.

## This run

| | |
|---|---|
| videos | {card['training']['videos']} ({n_test_videos} held out for test) |
| test AUC_gate | {m['headline_test']['auc_gate']:.4f} |
| test frames skipped | {m['headline_test']['frames_skipped_pct']:.1f}% (false skips {m['headline_test']['false_skip_pct']:.1f}%) |
| test REFIXATION recall | {m['headline_test']['refixation_recall']:.3f} |
| CV R2 (last 5 epochs) | {m['cv_last5']['r2_mean']:+.3f} +/- {m['cv_last5']['r2_std']:.3f} |
| L_sim only (ablation) R2 | {m['sim_only_ablation']['r2']:+.3f} |
| shuffle control AUC / R2 | {m['shuffle_control']['auc_gate']:.3f} / {m['shuffle_control']['r2']:+.3f} |

Thresholds are percentiles of the training data, not calibrated against a VLM. Every metric here
measures agreement with the DINOv2 similarity proxy.
"""

open(os.path.join(RELEASE, "example_inference.py"), "w").write(EXAMPLE)
open(os.path.join(RELEASE, "README.md"), "w").write(README)
if PERSIST:
    shutil.copytree(RELEASE, os.path.join(PERSIST, "release", "gate_v1"), dirs_exist_ok=True)
    print(f"copied the release to {PERSIST}/release/gate_v1")

tot = 0
for f in sorted(os.listdir(RELEASE)):
    sz = os.path.getsize(os.path.join(RELEASE, f)); tot += sz
    print(f"   {f:<26} {sz/1e6:7.2f} MB")
print(f"   {'TOTAL':<26} {tot/1e6:7.2f} MB")
print('\nto use it anywhere:  gate = torch.jit.load("gate_ensemble.ts.pt").eval();  pred = gate(rates, mask)')